In [0]:
%sql
create or replace table db(
    product_id int,
    product_name string,
    price int
)
using delta;

In [0]:
%sql
insert into db (product_id, product_name, price) values
    (101,"Laptop",50000),
    (102,"Mobile",20000),
    (103,"Tablet",10000);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from db;

product_id,product_name,price
101,Laptop,50000
102,Mobile,20000
103,Tablet,10000


In [0]:
data = [
    (4, "Monitor", 12000),
    (5, "Printer", 18000),
    (6, "Webcam", 2500)
]
 
df = spark.createDataFrame(
    data,
    ["product_id", "product_name", "price"]
)
 
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("retail_products_pyspark")
 
display(spark.table("retail_products_pyspark"))

product_id,product_name,price
4,Monitor,12000
5,Printer,18000
6,Webcam,2500


In [0]:
%sql
CREATE OR REPLACE TABLE retail_products_ctas
USING DELTA
AS
SELECT *
FROM VALUES
    (7, 'Tablet', 25000),
    (8, 'Speaker', 5000),
    (9, 'Router', 3500)
AS products(product_id, product_name, price);

num_affected_rows,num_inserted_rows


In [0]:
%sql
select * from retail_products_ctas;

product_id,product_name,price
7,Tablet,25000
8,Speaker,5000
9,Router,3500


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW product_updates AS
SELECT *
FROM VALUES
    (2, 'Wireless Mouse', 950),
    (10, 'USB Hub', 1200)
AS updates(product_id, product_name, price);

In [0]:
%sql
MERGE INTO db AS target
USING product_updates AS source
ON target.product_id = source.product_id

WHEN MATCHED THEN
    UPDATE SET
        target.product_name = source.product_name,
        target.price = source.price

WHEN NOT MATCHED THEN
    INSERT (product_id, product_name, price)
    VALUES (source.product_id, source.product_name, source.price);
        

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,0,0,2


In [0]:
%sql
SELECT * FROM db
ORDER BY product_id;

product_id,product_name,price
2,Wireless Mouse,950
10,USB Hub,1200
101,Laptop,50000
102,Mobile,20000
103,Tablet,10000


In [0]:
%sql
describe detail db;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,2f792622-3d29-4c2a-9002-853f25dd8f5f,databricks_kishore.default.db,null,abfss://unity-catalog-storage@dbstorage6jp3mgfxmdfz6.dfs.core.windows.net/7405616218552232/__unitystorage/catalogs/356a6522-af45-4662-aa20-00cd72ee4fac/tables/139491f9-92fc-4e37-bcbe-6d5345c9b58c,2026-11-05T07:22:55.933Z,2026-11-05T08:50:35Z,List(),List(),3,3840,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-ca7ba55d-b998-45e9-bf91-12f0537e4e34, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-6e848dcf-444d-4e2c-9c7f-7e6f719a40b2)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
describe history db;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-11-05T08:50:35Z,144786204307274,azuser7404_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(product_id#1255 = product_id#1246)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(293039832217956),null,1105-071633-x605gwbf,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 2549, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 6100, materializeSourceTimeMs -> 232, numTargetRowsInserted -> 2, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 2412, numTargetRowsUpdated -> 0, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 3403)",null,Databricks-Runtime/18.x-photon-scala2.13
1,2026-11-05T07:24:57Z,144786204307274,azuser7404_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(293039832217956),null,1105-071633-x605gwbf,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1291)",null,Databricks-Runtime/18.x-photon-scala2.13
0,2026-11-05T07:22:56Z,144786204307274,azuser7404_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-ca7ba55d-b998-45e9-bf91-12f0537e4e34"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-6e848dcf-444d-4e2c-9c7f-7e6f719a40b2""}, statsOnLoad -> false)",null,List(293039832217956),null,1105-071633-x605gwbf,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-photon-scala2.13


In [0]:
%sql
select * from db version as of 2;

product_id,product_name,price
2,Wireless Mouse,950
101,Laptop,50000
102,Mobile,20000
103,Tablet,10000
10,USB Hub,1200


In [0]:
%sql
optimize db;

path,metrics
abfss://unity-catalog-storage@dbstorage6jp3mgfxmdfz6.dfs.core.windows.net/7405616218552232/__unitystorage/catalogs/356a6522-af45-4662-aa20-00cd72ee4fac/tables/139491f9-92fc-4e37-bcbe-6d5345c9b58c,"List(1, 3, List(2192, 2192, 2192.0, 1, 2192), List(1256, 1293, 1280.0, 3, 3840), 0, null, null, 0, 1, 3, 0, true, 0, 0, 1793869624547, 1793869627899, 8, 1, null, List(0, 0), null, 3, 3, 516, 0, null, null)"


In [0]:
%sql
optimize db zorder by (product_id);

path,metrics
abfss://unity-catalog-storage@dbstorage6jp3mgfxmdfz6.dfs.core.windows.net/7405616218552232/__unitystorage/catalogs/356a6522-af45-4662-aa20-00cd72ee4fac/tables/139491f9-92fc-4e37-bcbe-6d5345c9b58c,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 2192), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1793870054361, 1793870055082, 8, 0, null, List(0, 0), null, 3, 3, 0, 0, null, null)"
